# SEG Multi-Seed Benchmark — QM8

Kompaktes Notebook für Experimente mit verschiedenen Seeds.
Multi-task Regression (16 elektronische Anregungseigenschaften).

**Workflow:**
1. Zellen 1-4 einmal ausführen (Setup, Daten, Embeddings, Funktionen)  
2. Config definieren und `run_single_seed(config, seed)` aufrufen  
3. Oder: `run_multi_seed(config, seeds=[...])` für aggregierte Ergebnisse

In [1]:
# === Setup (einmal ausführen) ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch
import deepchem as dc
from typing import Dict, List, Any, Optional

print(f"Workspace: {workspace_root}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

Workspace: c:\Users\robsc\Home\Dev\molfusion2
PyTorch: 2.6.0+cu124, CUDA: True


In [2]:
# === Default Configuration ===
DEFAULT_CONFIG = {
    # Architecture
    "hidden_channels": 128,
    "K": 4,
    "num_layers": 3,
    "pool": "set2set",
    "set2set_processing_steps": 6,
    
    # Fusion
    "fusion": "cross_mha",
    "fusion_dim": 64,
    "fusion_n_heads": 8,
    #"text_projection_dim": 64,
    "text_proj_init": "xavier",
    "text_proj_init_gain": 0.1,
    "freeze_text_proj": False,
    
    # Regularization
    "dropout": 0.3,
    #"fusion_dropout": 0.3,
    #"head_dropout": 0.6,
    "weight_decay": 1e-1,
    
    # Head
    "head_type": "mlp",
    "head_hidden_dim": 32,
    
    # Training
    "learning_rate": 1e-3,
    "batch_size": 64,
    "num_epochs": 100,
    "patience": 15,
    "scheduler": "cosine",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    "grad_clip": None,
    
    # Multi-task regression
    "num_tasks": 16,
}

print("Default config loaded.")
print(f"Tasks: {DEFAULT_CONFIG['num_tasks']} electronic properties")

Default config loaded.
Tasks: 16 electronic properties


In [3]:
# === Load QM8 Dataset (einmal ausführen) ===
SPLIT_TYPE = "random"  # Options: "scaffold" or "random"

# Load QM8 with NormalizationTransformer
tasks, datasets, transformers = dc.molnet.load_qm8(
    featurizer='ECFP',
    splitter='scaffold' if SPLIT_TYPE == 'scaffold' else 'random',
)
train_dc, valid_dc, test_dc = datasets
QM8_TASKS = tasks  # 16 electronic properties

# Extract normalized data (for training)
TRAIN_SMILES = list(train_dc.ids)
TRAIN_Y = train_dc.y.astype(np.float32)
VALID_SMILES = list(valid_dc.ids)
VALID_Y = valid_dc.y.astype(np.float32)
TEST_SMILES = list(test_dc.ids)
TEST_Y = test_dc.y.astype(np.float32)

# Store original (un-normalized) labels for evaluation
TRAIN_Y_ORIG = TRAIN_Y.copy()
VALID_Y_ORIG = VALID_Y.copy()
TEST_Y_ORIG = TEST_Y.copy()
for transformer in reversed(transformers):
    TRAIN_Y_ORIG = transformer.untransform(TRAIN_Y_ORIG)
    VALID_Y_ORIG = transformer.untransform(VALID_Y_ORIG)
    TEST_Y_ORIG = transformer.untransform(TEST_Y_ORIG)
TRAIN_Y_ORIG = TRAIN_Y_ORIG.astype(np.float32)
VALID_Y_ORIG = VALID_Y_ORIG.astype(np.float32)
TEST_Y_ORIG = TEST_Y_ORIG.astype(np.float32)

# Store transformers globally for inverse-transform in evaluation
TRANSFORMERS = transformers

print(f"Split: {SPLIT_TYPE} | Train: {len(TRAIN_SMILES)} | Valid: {len(VALID_SMILES)} | Test: {len(TEST_SMILES)}")
print(f"Train labels shape: {TRAIN_Y.shape}")
print(f"Transformers: {[type(t).__name__ for t in transformers]}")
print(f"Training on NORMALIZED data; will inverse-transform predictions for MAE evaluation")

Split: random | Train: 17397 | Valid: 2175 | Test: 2175
Train labels shape: (17397, 16)
Transformers: ['NormalizationTransformer']
Training on NORMALIZED data; will inverse-transform predictions for MAE evaluation


In [4]:
# === Load Text Embeddings (einmal ausführen) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_EMB_DIR = workspace_root / "cache" / "cot_embeddings"
TASK = "quantum_fast"

npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = TRAIN_SMILES + VALID_SMILES + TEST_SMILES
all_emb = torch.from_numpy(cache.get_batch(all_smiles))

n_train, n_valid = len(TRAIN_SMILES), len(VALID_SMILES)
TRAIN_TEXT_EMB = all_emb[:n_train]
VALID_TEXT_EMB = all_emb[n_train:n_train + n_valid]
TEST_TEXT_EMB = all_emb[n_train + n_valid:]

print(f"\u2713 Loaded {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={TRAIN_TEXT_EMB.shape}, valid={VALID_TEXT_EMB.shape}, test={TEST_TEXT_EMB.shape}")

Loading embeddings from quantum_fast_text_embeddings_compact.npz...
  Loaded 21722 entries, dim=3072
  Memory mode: mapped
✓ Loaded quantum_fast_text_embeddings_compact.npz (21722 molecules)
  Embeddings: train=torch.Size([17397, 3072]), valid=torch.Size([2175, 3072]), test=torch.Size([2175, 3072])


In [5]:
# === Experiment Functions (einmal ausführen) ===
from itertools import product
from models import SEGPredictor, SEGPredictorConfig


def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def multitask_mae(y_true, y_pred):
    """Per-task MAE, returns mean MAE and per-task MAEs."""
    maes = []
    for i in range(y_true.shape[1]):
        mask = ~np.isnan(y_true[:, i]) & ~np.isnan(y_pred[:, i])
        if mask.sum() > 0:
            maes.append(float(np.mean(np.abs(y_true[mask, i] - y_pred[mask, i]))))
        else:
            maes.append(np.nan)
    valid_maes = [m for m in maes if not np.isnan(m)]
    mean_mae = float(np.mean(valid_maes)) if valid_maes else np.nan
    return {"mean_mae": mean_mae, "per_task_maes": maes, "n_valid_tasks": len(valid_maes)}


def run_single_seed(
    config: Dict[str, Any],
    seed: int,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run a single SEG experiment with the given config and seed.
    Trains on normalized labels, evaluates MAE on original (Hartree) scale.
    """
    cfg = {**DEFAULT_CONFIG, **config}
    set_seed(seed)
    
    seg_config = SEGPredictorConfig(
        task="regression",
        num_tasks=cfg["num_tasks"],
        hidden_channels=cfg["hidden_channels"],
        K=cfg["K"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        pool=cfg["pool"],
        set2set_processing_steps=cfg["set2set_processing_steps"],
        text_embedding_dim=3072,
        text_projection_dim=cfg["fusion_dim"],
        text_proj_init=cfg["text_proj_init"],
        text_proj_init_gain=cfg["text_proj_init_gain"],
        freeze_text_proj=cfg["freeze_text_proj"],
        fusion=cfg["fusion"],
        fusion_dim=cfg["fusion_dim"],
        fusion_n_heads=cfg.get("fusion_n_heads", 8),
        fusion_dropout=cfg["dropout"],
        head_type=cfg["head_type"],
        head_hidden_dim=cfg["head_hidden_dim"],
        head_dropout=cfg["dropout"],
    )
    
    seg = SEGPredictor(config=seg_config)
    
    if verbose:
        print(f"=== Seed {seed} ===")
    
    history = seg.fit(
        smiles_list=TRAIN_SMILES,
        labels=TRAIN_Y,
        val_smiles=VALID_SMILES,
        val_labels=VALID_Y,
        text_embeddings=TRAIN_TEXT_EMB,
        val_text_embeddings=VALID_TEXT_EMB,
        num_epochs=cfg["num_epochs"],
        batch_size=cfg["batch_size"],
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        patience=cfg["patience"],
        scheduler=cfg["scheduler"],
        scheduler_patience=cfg["scheduler_patience"],
        scheduler_factor=cfg["scheduler_factor"],
        min_lr=cfg["min_lr"],
        grad_clip=cfg["grad_clip"],
        seed=seed,
        verbose=verbose,
    )
    
    # Predict (normalized) and inverse-transform to original scale
    test_preds_norm = seg.predict_batch(TEST_SMILES, text_embeddings=TEST_TEXT_EMB)
    test_preds = test_preds_norm.copy()
    for transformer in reversed(TRANSFORMERS):
        test_preds = transformer.untransform(test_preds)
    test_preds = test_preds.astype(np.float32)
    
    metrics = multitask_mae(TEST_Y_ORIG, test_preds)
    
    if verbose:
        print(f"\u2192 Test Mean MAE={metrics['mean_mae']*1000:.4f}\u00d710\u207b\u00b3 ({metrics['n_valid_tasks']} tasks)\n")
    
    return {"seed": seed, "metrics": metrics, "history": history, "model": seg}


def run_multi_seed(
    config: Dict[str, Any],
    seeds: List[int],
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Run experiments with multiple seeds and aggregate results.
    """
    results = []
    
    print(f"Running {len(seeds)} experiments with seeds: {seeds}")
    print("-" * 60)
    
    for i, seed in enumerate(seeds):
        print(f"[{i+1}/{len(seeds)}] Seed {seed}...", end=" ", flush=True)
        result = run_single_seed(config, seed, verbose=verbose)
        results.append(result)
        if not verbose:
            m = result["metrics"]
            print(f"Mean MAE={m['mean_mae']*1000:.4f}\u00d710\u207b\u00b3")
    
    df = pd.DataFrame([
        {"seed": r["seed"], "mean_mae": r["metrics"]["mean_mae"], "n_valid_tasks": r["metrics"]["n_valid_tasks"]}
        for r in results
    ])
    
    summary = {
        "mean_mae_mean": df["mean_mae"].mean(),
        "mean_mae_std": df["mean_mae"].std(),
        "n_runs": len(seeds),
        "seeds": seeds,
    }
    
    print("-" * 60)
    print(f"\n=== AGGREGATED RESULTS ({len(seeds)} seeds) ===")
    print(f"Mean MAE: {summary['mean_mae_mean']*1000:.4f} \u00b1 {summary['mean_mae_std']*1000:.4f} (\u00d710\u207b\u00b3)")
    
    return {"summary": summary, "runs": results, "df": df}


def run_grid_search(
    grid_config: Dict[str, Any],
    seeds: List[int] = [42],
    sort_by: str = "mean_mae",
    ascending: bool = True,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Grid search over hyperparameters with multi-seed evaluation.
    
    Values that are lists \u2192 swept over (all combinations).
    Values that are scalars \u2192 fixed for all runs.
    """
    sweep_keys = []
    sweep_values = []
    fixed_params = {}
    
    for k, v in grid_config.items():
        if isinstance(v, list):
            sweep_keys.append(k)
            sweep_values.append(v)
        else:
            fixed_params[k] = v
    
    if sweep_keys:
        combos = list(product(*sweep_values))
    else:
        combos = [()]
    
    n_combos = len(combos)
    n_total = n_combos * len(seeds)
    
    print(f"=== GRID SEARCH ===")
    if sweep_keys:
        print(f"Sweep params: {', '.join(f'{k} ({len(v)} values)' for k, v in zip(sweep_keys, sweep_values))}")
    else:
        print("No sweep params (single config)")
    print(f"Combinations: {n_combos} \u00d7 {len(seeds)} seeds = {n_total} total runs")
    print("=" * 70)
    
    all_rows = []
    run_counter = 0
    
    for combo_idx, combo in enumerate(combos):
        config = {**fixed_params}
        for k, v in zip(sweep_keys, combo):
            config[k] = v
        
        combo_desc = ", ".join(f"{k}={v}" for k, v in zip(sweep_keys, combo)) if sweep_keys else "default"
        print(f"\n[{combo_idx+1}/{n_combos}] {combo_desc}")
        
        seed_metrics = []
        for seed in seeds:
            run_counter += 1
            print(f"  ({run_counter}/{n_total}) seed={seed}...", end=" ", flush=True)
            result = run_single_seed(config, seed, verbose=verbose)
            seed_metrics.append(result["metrics"])
            m = result["metrics"]
            print(f"MAE={m['mean_mae']*1000:.4f}\u00d710\u207b\u00b3")
        
        maes = [m["mean_mae"] for m in seed_metrics]
        
        row = {**{k: v for k, v in zip(sweep_keys, combo)}}
        row["mean_mae_mean"] = np.mean(maes)
        row["mean_mae_std"] = np.std(maes)
        row["n_seeds"] = len(seeds)
        all_rows.append(row)
    
    df = pd.DataFrame(all_rows)
    
    sort_col = f"{sort_by}_mean"
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
    
    print("\n" + "=" * 70)
    print(f"GRID SEARCH RESULTS (sorted by {sort_by}, lower is better)")
    print("=" * 70)
    
    for i, row in df.iterrows():
        params = " | ".join(f"{k}={row[k]}" for k in sweep_keys) if sweep_keys else "default"
        print(f"  #{i+1}: {params}")
        print(f"      Mean MAE={row['mean_mae_mean']*1000:.4f}\u00b1{row['mean_mae_std']*1000:.4f} (\u00d710\u207b\u00b3)")
    
    return df


print("\u2713 Functions loaded: run_single_seed, run_multi_seed, run_grid_search")

✓ Functions loaded: run_single_seed, run_multi_seed, run_grid_search


---
## Experimente

Ab hier: Config definieren und Experimente starten.

In [6]:
# === Einzelnes Experiment ===
# Config-Overrides (leer = Default-Config verwenden)
my_config = {
    # Hier eigene Werte überschreiben, z.B.:
    # "dropout": 0.5,
    # "weight_decay": 5e-2,
}

#result = run_single_seed(my_config, seed=42)

In [7]:
# === Multi-Seed Experiment ===
my_config = {}  # Default-Config

#results = run_multi_seed(my_config, seeds=[42, 123, 456, 789, 1337])

In [8]:
# === Ergebnisse anzeigen ===
#results["df"]

---
## Varianten testen

Config anpassen und erneut ausführen:

In [9]:
# === Grid Search Beispiel ===
# Werte als Liste → werden gesweept (alle Kombinationen)
# Werte als Skalar → bleiben fix
grid_config = {
    "hidden_channels" : [64, 128, 256],
    "K" : [3, 4, 5],
    "text_proj_init": ["xavier"],
    "pool": ["sum"],
    #"freeze_text_proj": [True, False],
    #"fusion_dim": [32, 64],
    "fusion": "cross_mha",       # fix
    "head_type": "mlp",          # fix
}

# Grid search mit 4 Seeds pro Kombination
grid_df = run_grid_search(grid_config, seeds=[42, 113])

=== GRID SEARCH ===
Sweep params: hidden_channels (3 values), K (3 values), text_proj_init (1 values), pool (1 values)
Combinations: 9 × 2 seeds = 18 total runs

[1/9] hidden_channels=64, K=3, text_proj_init=xavier, pool=sum
  (1/18) seed=42... 

[20:01:25] WARNING: not removing hydrogen atom without neighbors
[20:01:25] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not removing hydrogen atom without neighbors
[20:01:26] WARNING: not r

MAE=34.0303×10⁻³
  (2/18) seed=113... 

[20:05:45] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:46] WARNING: not removing hydrogen atom without neighbors
[20:05:47] WARNING: not removing hydrogen atom without neighbors
[20:05:47] WARNING: not removing hydrogen atom without neighbors
[20:05:47] WARNING: not removing hydrogen atom without neighbors
[20:05:47] WARNING: not removing hydrogen atom without neighbors
[20:05:47] WARNING: not r

MAE=34.4115×10⁻³

[2/9] hidden_channels=64, K=4, text_proj_init=xavier, pool=sum
  (3/18) seed=42... 

...

MAE=31.0674×10⁻³

[9/9] hidden_channels=256, K=5, text_proj_init=xavier, pool=sum
  (17/18) seed=42... 

[21:25:21] WARNING: not removing hydrogen atom without neighbors
[21:25:21] WARNING: not removing hydrogen atom without neighbors
[21:25:21] WARNING: not removing hydrogen atom without neighbors
[21:25:21] WARNING: not removing hydrogen atom without neighbors
[21:25:21] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not removing hydrogen atom without neighbors
[21:25:22] WARNING: not r

MAE=31.3144×10⁻³
  (18/18) seed=113... 

[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:39] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not removing hydrogen atom without neighbors
[21:31:40] WARNING: not r

MAE=31.3204×10⁻³

GRID SEARCH RESULTS (sorted by mean_mae, lower is better)
  #1: hidden_channels=256 | K=4 | text_proj_init=xavier | pool=sum
      Mean MAE=31.0777±0.0103 (×10⁻³)
  #2: hidden_channels=256 | K=5 | text_proj_init=xavier | pool=sum
      Mean MAE=31.3174±0.0030 (×10⁻³)
  #3: hidden_channels=128 | K=5 | text_proj_init=xavier | pool=sum
      Mean MAE=31.8129±0.0569 (×10⁻³)
  #4: hidden_channels=256 | K=3 | text_proj_init=xavier | pool=sum
      Mean MAE=32.0537±0.5143 (×10⁻³)
  #5: hidden_channels=128 | K=4 | text_proj_init=xavier | pool=sum
      Mean MAE=32.2789±0.1642 (×10⁻³)
  #6: hidden_channels=64 | K=5 | text_proj_init=xavier | pool=sum
      Mean MAE=32.9218±0.1593 (×10⁻³)
  #7: hidden_channels=128 | K=3 | text_proj_init=xavier | pool=sum
      Mean MAE=33.1676±0.1129 (×10⁻³)
  #8: hidden_channels=64 | K=4 | text_proj_init=xavier | pool=sum
      Mean MAE=33.3454±0.0889 (×10⁻³)
  #9: hidden_channels=64 | K=3 | text_proj_init=xavier | pool=sum
      Mean MAE=34.22

In [10]:
# === Grid Search Ergebnisse ===
grid_df

,hidden_channels,K,text_proj_init,pool,mean_mae_mean,mean_mae_std,n_seeds
0,256,4,xavier,sum,0.031078,0.000010,2
1,256,5,xavier,sum,0.031317,0.000003,2
2,128,5,xavier,sum,0.031813,0.000057,2
3,256,3,xavier,sum,0.032054,0.000514,2
4,128,4,xavier,sum,0.032279,0.000164,2
5,64,5,xavier,sum,0.032922,0.000159,2
6,128,3,xavier,sum,0.033168,0.000113,2
7,64,4,xavier,sum,0.033345,0.000089,2
8,64,3,xavier,sum,0.034221,0.000191,2


In [ ]:
# === Grid Search Ergebnisse speichern ===
out_path = workspace_root / "benchmarking" / "results" / "grid_search_qm8_2.csv"
grid_df.to_csv(out_path, index=False)
print(f"\u2713 Saved to {out_path}")

✓ Saved to c:\Users\robsc\Home\Dev\molfusion2\benchmarking\results\grid_search_qm8_1.csv
